# **Tech Challenge - 3**  👨🏻‍💻

**Problema:**

*   Executar o fine-tuning de um foundation model (Llama, BERT, MISTRAL etc.)

# 🔧 Instalação de Dependências

Estas células instalam as principais bibliotecas necessárias para o fine-tuning do modelo Llama 3 utilizando o framework Unsloth e outras ferramentas de otimização e avaliação.

- `unsloth[colab-new]`: Framework para aceleração e fine-tuning de modelos LLM. Instalado diretamente do repositório oficial via Git.
- `xformers`: Biblioteca para operações eficientes em transformers (atenção, restrição de versão para evitar incompatibilidades).
- `peft`, `accelerate`, `bitsandbytes`: Ferramentas para treinamento eficiente, gerenciamento de parâmetros e quantização.
- `evaluate`, `datasets`, `rouge-score`: Utilitários para avaliação de modelos e manipulação de datasets.

In [ ]:
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install no-deps "xformers<0.0.28" peft accelerate bitsandbytes
%pip install evaluate datasets rouge-score

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ledm5hbl/unsloth_aeb53252e0da4015bbf6369dfec7859d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ledm5hbl/unsloth_aeb53252e0da4015bbf6369dfec7859d
  Resolved https://github.com/unslothai/unsloth.git to commit 79ae18052c939578f303ba2f7823dae03cb00b5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 💾 Montagem do Google Drive e Definição do Diretório de Saída

Neste bloco, o Google Drive é montado no ambiente do Colab, permitindo o armazenamento persistente dos checkpoints do modelo. O diretório `output_dir` é definido para salvar os checkpoints do treinamento diretamente na pasta desejada do Google Drive.

In [2]:
from google.colab import drive
drive.mount("/content/drive")
output_dir = "/content/drive/MyDrive/llama_unsloth_checkpoints"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Este bloco carrega o modelo Llama 3.2-3B em 4 bits usando Unsloth, define parâmetros de LoRA para fine-tuning eficiente e inicializa o tokenizer.

# 🦙 Carregamento e Configuração do Modelo Llama 3.2 3B (Unsloth)

Este bloco carrega o modelo Llama 3.2-3B em modo 4-bit, utilizando o framework Unsloth para otimização de performance e memória. Em seguida, aplica a técnica de PEFT (Parameter-Efficient Fine-Tuning) via LoRA, configurando módulos e parâmetros que impactam o desempenho do fine-tuning.


## ⚙️ Documentação dos Parâmetros Utilizados

- **max_seq_length = 1024**
  - Define o tamanho máximo de sequência de tokens processados pelo modelo.  
  - Valor maior permite processar textos mais longos, porém aumenta o uso de memória e tempo de processamento.

- **dtype = None**
  - Tipo de dado utilizado nos pesos do modelo.  
  - Se definido como None, o tipo padrão do modelo será utilizado.  
  - Normalmente pode ser ajustado para `torch.float16` ou `torch.bfloat16` para otimização de memória.

- **load_in_4bit = True**
  - Permite carregar o modelo em modo quantizado de 4 bits, reduzindo drasticamente o consumo de memória sem grandes perdas de desempenho.

- **r = 16**
  - Hiperparâmetro da LoRA, define o rank do fator de decomposição para as matrizes de adaptação.  
  - Valores maiores aumentam a capacidade de adaptação, mas também o uso de memória.

- **target_modules**
  - Lista de módulos do modelo onde a LoRA será aplicada.  
  - Inclui projeções de atenção e feedforward: `"q_proj"`, `"k_proj"`, `"v_proj"`, `"o_proj"`, `"gate_proj"`, `"up_proj"`, `"down_proj"`.

- **lora_alpha = 16**
  - Define o fator de escala para os módulos LoRA, impactando o quanto as adaptações influenciam o modelo.

- **lora_dropout = 0**
  - Taxa de dropout aplicada nas camadas LoRA.  
  - Valor zero significa que não será aplicado dropout, favorecendo adaptação total.

- **bias = "none"**
  - Configura se os termos de bias (viés) serão adaptados ou não durante o fine-tuning.  
  - `"none"` indica que não haverá adaptação dos termos de bias.

- **use_gradient_checkpointing = "unsloth"**
  - Habilita o gradient checkpointing para otimizar o uso de memória durante o treinamento, permitindo treinar modelos maiores com menos recursos.

- **random_state = 3407**
  - Define a seed (semente) para geração de números aleatórios, garantindo reprodutibilidade dos resultados do treinamento.

---

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3.2-3B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.4.0+cu121 with CUDA 1201 (you have 2.8.0+cu126)
    Python  3.12.4 (you have 3.12.11)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.4.0+cu121 with CUDA 1201 (you have 2.8.0+cu126)
    Python  3.12.4 (you have 3.12.11)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.9: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/u

Unsloth 2025.9.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 📦 Pré-processamento do Dataset

Este bloco é responsável por carregar, transformar e preparar o dataset para o fine-tuning do modelo.

**Passos realizados:**

1. **Carregamento do Dataset**
   - Utiliza o HuggingFace `datasets` para carregar o dataset `"guillherms/amazon_titles_alpaca_cleaned_v2"`, que contém instruções, entradas e saídas para treinamento e validação.

2. **Conversão para Formato ShareGPT**
   - Função `to_sharegpt` transforma os dados para o formato esperado pelo modelo, usando um prompt que une instrução e entrada:
     - `merged_prompt="{Instruction}[[\nYour input is:\n{Input}]]"` faz a junção dos campos.
     - `output_column_name="Output"` indica qual coluna contém a resposta esperada.
     - `conversation_extension=3` permite extensão de contexto de conversa.

3. **Seleção e Embaralhamento**
   - Para garantir diversidade, os datasets de treino e validação são embaralhados (shuffle) com seed fixa para reprodutibilidade.
   - Seleciona exatamente 100.000 exemplos para treino e 5.000 para validação.

4. **Padronização e Template**
   - `standardize_sharegpt`: Padroniza estrutura dos exemplos para o formato ShareGPT.
   - `apply_chat_template`: Aplica o template de chat esperado pelo modelo e tokenizador.

5. **Divisão em blocos**
   - Divide o dataset de treino em 10 blocos de 10.000 exemplos cada para facilitar o processamento e o treinamento em etapas.


In [ ]:
from datasets import load_dataset
from unsloth import to_sharegpt, standardize_sharegpt, apply_chat_template

raw_dataset = load_dataset("guillherms/amazon_titles_alpaca_cleaned_v2")

train_dataset = to_sharegpt(
    raw_dataset["train"],
    merged_prompt="{Instruction}[[\nYour input is:\n{Input}]]",
    output_column_name="Output",
    conversation_extension=3,
)
val_dataset = to_sharegpt(
    raw_dataset["validation"],
    merged_prompt="{Instruction}[[\nYour input is:\n{Input}]]",
    output_column_name="Output",
    conversation_extension=3,
)

# Selecionar exatamente 100k para treino + 5k val
train_dataset = train_dataset.shuffle(seed=3407).select(range(100_000))
val_dataset   = val_dataset.shuffle(seed=3407).select(range(5000))

dataset = {"train": train_dataset, "validation": val_dataset}
dataset = {k: standardize_sharegpt(v) for k, v in dataset.items()}
dataset = {k: apply_chat_template(v, tokenizer=tokenizer) for k, v in dataset.items()}

print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

# Dividir em 10 blocos de 10k para evitar estouro de memória
blocks = [dataset["train"].select(range(i*10_000, (i+1)*10_000)) for i in range(10)]

Unsloth: We automatically added an EOS token to stop endless generations.


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Unsloth: We automatically added an EOS token to stop endless generations.


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train size: 100000
Validation size: 5000


## 🏋️‍♂️ Função de Treinamento em Blocos e Gerenciamento de Checkpoints

Este bloco define funções essenciais para o treinamento incremental do modelo, utilizando checkpoints para retomar ou continuar o treinamento e garantindo o uso eficiente de memória.

### Funções e Lógica

- **get_last_checkpoint(output_dir)**
  - Busca o último checkpoint salvo no diretório de saída, permitindo retomar o treinamento de onde parou.
  - Retorna o caminho do último checkpoint ou None se não houver checkpoints.

- **train_block(block_id, dataset_block)**
  - Realiza o treinamento de um bloco específico do dataset.
  - Gerencia memória (coleta de lixo e limpeza da GPU).
  - Detecta e recarrega checkpoint mais recente, reconstruindo o modelo e carregando adapters LoRA.
  - Se não houver checkpoint, inicia treinamento do zero.
  - Treina usando `SFTTrainer` da biblioteca `trl`, com configurações ajustadas para eficiência.
  - Avalia o modelo após o treinamento do bloco, calcula métricas de loss e perplexidade.

### Parâmetros Importantes

- `output_dir`: Diretório onde checkpoints são salvos e buscados.
- `max_seq_length`: Máximo de tokens por exemplo.
- `packing=True`: Junta exemplos curtos para otimizar processamento.
- `per_device_train_batch_size=1`: Tamanho do batch por dispositivo.
- `gradient_accumulation_steps=2`: Passos de acumulação de gradiente.
- `learning_rate=2e-4`: Taxa de aprendizado.
- `num_train_epochs=1`: Número de épocas por bloco.
- `save_steps=1000`: Salva checkpoint a cada 1000 passos.
- `save_total_limit=2`: Mantém até 2 checkpoints.
- `optim="paged_adamw_8bit"`: Otimizador eficiente para memória.
- `seed=3407`: Garante reprodutibilidade.

### Exemplo de Uso

```python
metrics = train_block(block_id=0, dataset_block=blocks[0])
```

### Observações

- A função permite treinamento incremental, importante para grandes datasets.
- A avaliação por bloco facilita monitoramento do desempenho ao longo do treinamento.
- O uso de checkpoints garante segurança contra perda de progresso e viabiliza experimentos contínuos.

In [5]:
import os, gc
from trl import SFTConfig, SFTTrainer
from math import exp

def get_last_checkpoint(output_dir):
    ckpts = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not ckpts: return None
    ckpts = sorted(ckpts, key=lambda x: int(x.split("-")[1]))
    return os.path.join(output_dir, ckpts[-1])

def train_block(block_id, dataset_block):
    gc.collect(); torch.cuda.empty_cache()
    print(f"\n==== Treinando BLOCO {block_id+1}/10 ====")

    # Detectar último checkpoint e recarregar adapters
    last_ckpt = get_last_checkpoint(output_dir)
    if last_ckpt:
        print(f"🔄 Recriando modelo e carregando adapters de {last_ckpt}")
        base_model, tokenizer_ckpt = FastLanguageModel.from_pretrained(
            "unsloth/Llama-3.2-3B-bnb-4bit",
            max_seq_length=max_seq_length,
            dtype=None,
            load_in_4bit=True,
        )
        global model, tokenizer
        model = FastLanguageModel.get_peft_model(
            base_model,
            r=16,
            target_modules=["q_proj","k_proj","v_proj","o_proj",
                            "gate_proj","up_proj","down_proj"],
            lora_alpha=16,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=3407,
        )
        model.load_adapter(last_ckpt, adapter_name="default")
        model.train()
        tokenizer = tokenizer_ckpt
    else:
        print("🆕 Nenhum checkpoint encontrado — iniciando do zero.")
        model.train()

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset_block,
        eval_dataset=dataset["validation"],
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        packing=True,   # junta exemplos curtos → mais rápido
        args=SFTConfig(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=2,
            learning_rate=2e-4,
            num_train_epochs=1,
            logging_steps=50,
            save_steps=1000,
            save_total_limit=2,
            output_dir=output_dir,
            report_to="none",
            optim="paged_adamw_8bit",
            seed=3407,
        ),
    )

    trainer.train()
    metrics = trainer.evaluate()
    metrics["perplexity"] = exp(metrics["eval_loss"])
    print(f"✅ Bloco {block_id+1} finalizado -> Loss: {metrics['eval_loss']:.4f}, PPL: {metrics['perplexity']:.2f}")
    return metrics

## 📊 Avaliação das Gerações do Modelo

Este bloco implementa a avaliação automática das respostas geradas pelo modelo utilizando as métricas ROUGE e BLEU, comuns em tarefas de NLP para medir similaridade entre texto gerado e referência.

### Funcionalidades

- **Carregamento das Métricas**
  - `rouge` e `bleu` são carregados via biblioteca `evaluate` do HuggingFace.

- **Função `evaluate_generation`**
  - Gera respostas do modelo para um subconjunto do dataset (default: 200 exemplos).
  - Compara as respostas geradas (`preds`) com as referências reais (`refs`) presentes no dataset.
  - Suporta múltiplos formatos de referência (`output`, `Output`, `response`, `messages`).
  - Garante que listas de previsões e referências estejam alinhadas em tamanho.
  - Calcula e retorna as métricas ROUGE-L e BLEU.

### Parâmetros

- `model`: Modelo treinado para geração de texto.
- `tokenizer`: Tokenizador do modelo.
- `ds`: Dataset a ser avaliado.
- `n_samples`: Número de exemplos a serem amostrados para avaliação (default: 200).

### Fluxo de Avaliação

1. Seleciona até `n_samples` exemplos do dataset.
2. Para cada exemplo, gera a resposta usando o modelo.
3. Decodifica a resposta do modelo.
4. Busca e armazena a referência correta do exemplo, suportando diferentes formatos de campo.
5. Ajusta tamanho das listas de previsões e referências para alinhamento.
6. Computa as métricas ROUGE-L e BLEU.
7. Retorna um dicionário com os resultados.

### Exemplo de Uso

```python
resultados = evaluate_generation(model, tokenizer, dataset["validation"], n_samples=200)
print(resultados)
```

### Observações

- ROUGE-L mede a similaridade baseada em subsequências comuns (útil para sumarização).
- BLEU mede a precisão de n-gramas (comum em tradução automática).
- O alinhamento das listas evita erros caso o dataset esteja incompleto.
- O fallback garante que sempre há referência, evitando falhas na avaliação.

In [6]:
from evaluate import load

rouge = load("rouge")
bleu  = load("bleu")

def evaluate_generation(model, tokenizer, ds, n_samples=200):
    sub = ds.select(range(min(n_samples, len(ds))))
    preds, refs = [], []

    for ex in sub:
        # Entrada já preparada no campo "text"
        inp = tokenizer(ex["text"], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=128,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(pred)

        # Captura referência com fallback
        if "output" in ex:
            refs.append(ex["output"])
        elif "Output" in ex:
            refs.append(ex["Output"])
        elif "response" in ex:
            refs.append(ex["response"])
        elif "messages" in ex and len(ex["messages"]) > 0:
            refs.append(ex["messages"][-1]["content"])
        else:
            refs.append("")  # fallback → nunca vazio

    # Garantir alinhamento
    if len(preds) != len(refs):
        min_len = min(len(preds), len(refs))
        preds, refs = preds[:min_len], refs[:min_len]

    rouge_scores = rouge.compute(predictions=preds, references=refs)
    bleu_score   = bleu.compute(predictions=preds, references=[[r] for r in refs])

    return {"rougeL": rouge_scores["rougeL"], "bleu": bleu_score["bleu"]}

# 📈 Treinamento, Avaliação e Salvamento de Checkpoints por Bloco

Este bloco executa o loop principal de treinamento e avaliação, consolidando métricas e salvando snapshots do modelo após cada bloco.

---

### Fluxo de Execução

1. **Inicialização**
   - Cria lista `all_metrics` para armazenar métricas de cada bloco.

2. **Loop de Treinamento**
   - Para cada bloco do dataset:
     1. Treina o modelo usando a função `train_block`.
     2. Avalia o modelo com a função `evaluate_generation` usando 200 exemplos do conjunto de validação.
     3. Em caso de erro na avaliação, registra métricas nulas e exibe aviso.

3. **Armazenamento de Métricas**
   - Consolida métricas de treinamento e geração em um dicionário, adicionando à lista `all_metrics`.

4. **Salvamento de Checkpoint**
   - Salva o modelo e tokenizador após o término de cada bloco em um diretório nomeado por bloco.
   - Garante persistência dos resultados e possibilidade de retomar o treinamento.

5. **Resumo Final**
   - Converte a lista de métricas em um DataFrame para visualização e análise.
   - Imprime resumo da evolução dos resultados bloco a bloco.

---

### Parâmetros Principais

- `blocks`: Lista de blocos do dataset de treino (cada um com 10k exemplos).
- `train_block`: Função de treinamento definida anteriormente.
- `evaluate_generation`: Função de avaliação automática definida anteriormente.
- `snapshot_dir`: Diretório onde snapshots do modelo são salvos após cada bloco.
- `df_metrics`: DataFrame com métricas consolidadas para análise.

In [7]:
import pandas as pd
all_metrics = []

for i, block in enumerate(blocks):
    train_metrics = train_block(i, block)

    try:
        gen_metrics = evaluate_generation(model, tokenizer, dataset["validation"], n_samples=200)
    except Exception as e:
        print(f"⚠️ Erro na avaliação do bloco {i+1}: {e}")
        gen_metrics = {"rougeL": None, "bleu": None}

    row = {"block": i+1, **train_metrics, **gen_metrics}
    all_metrics.append(row)

    # Snapshot consolidado ao final do bloco
    snapshot_dir = f"/content/drive/MyDrive/llama_unsloth_checkpoints/final_lora_block{i+1}"
    model.save_pretrained(snapshot_dir)
    tokenizer.save_pretrained(snapshot_dir)
    print(f"💾 Snapshot salvo: {snapshot_dir}")

df_metrics = pd.DataFrame(all_metrics)
print("\nResumo da evolução por bloco:")
print(df_metrics)


==== Treinando BLOCO 1/10 ====
🔄 Recriando modelo e carregando adapters de /content/drive/MyDrive/llama_unsloth_checkpoints/checkpoint-5000
==((====))==  Unsloth 2025.9.9: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/10000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/5000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 5,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


...

✅ Bloco 9 finalizado -> Loss: 2.2882, PPL: 9.86
⚠️ Erro na avaliação do bloco 9: float division by zero
💾 Snapshot salvo: /content/drive/MyDrive/llama_unsloth_checkpoints/final_lora_block9

==== Treinando BLOCO 10/10 ====
🔄 Recriando modelo e carregando adapters de /content/drive/MyDrive/llama_unsloth_checkpoints/checkpoint-5000
==((====))==  Unsloth 2025.9.9: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/10000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 5,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
50,2.287300
100,2.269100
150,2.241600
200,2.219100
250,2.292000
300,2.275400
350,2.317500
400,2.272400
450,2.339800
500,2.304600


✅ Bloco 10 finalizado -> Loss: 2.2840, PPL: 9.82
⚠️ Erro na avaliação do bloco 10: float division by zero
💾 Snapshot salvo: /content/drive/MyDrive/llama_unsloth_checkpoints/final_lora_block10

Resumo da evolução por bloco:
   block  eval_loss  eval_runtime  eval_samples_per_second  \
0      1   2.383679     1207.0247                    4.142   
1      2   2.352355     1206.3536                    4.145   
2      3   2.336924     1206.5342                    4.144   
3      4   2.324496     1207.1375                    4.142   
4      5   2.314371     1206.5430                    4.144   
5      6   2.307032     1207.4641                    4.141   
6      7   2.300257     1206.5925                    4.144   
7      8   2.294358     1206.3015                    4.145   
8      9   2.288159     1205.2677                    4.148   
9     10   2.283958     1206.0502                    4.146   

   eval_steps_per_second  epoch  perplexity rougeL  bleu  
0                  1.036    1.0   1

# 🚀 Salvamento de Métricas e Publicação do Modelo na HuggingFace

Este bloco finaliza o pipeline salvando as métricas de treinamento e publicando o modelo treinado na HuggingFace Hub.

---

### Passos Realizados

1. **Salvar Métricas em CSV**
   - Exporta todas as métricas consolidadas do DataFrame (`df_metrics`) para um arquivo CSV.
   - Garante que os resultados de cada bloco fiquem disponíveis para consulta e análise futura.

2. **Copiar CSV para Pasta Final do Modelo**
   - Move o arquivo de métricas para o diretório do último bloco treinado (`final_lora_block10`).
   - Facilita o upload do modelo juntamente das métricas.

3. **Login na HuggingFace**
   - Recupera o token de autenticação via Colab para login automático na HuggingFace Hub.

4. **Criação e Upload do Repositório**
   - Cria (ou utiliza) o repositório público na HuggingFace para armazenar o modelo treinado.
   - Faz upload do conteúdo da pasta final do modelo, incluindo checkpoints e métricas.

5. **Mensagem de Publicação**
   - Exibe o link público do repositório HuggingFace após o upload bem-sucedido.

In [8]:
import os
from google.colab import userdata
from huggingface_hub import login, create_repo, upload_folder



# 1. Salvar métricas em CSV
metrics_csv = "/content/drive/MyDrive/llama_unsloth_checkpoints/training_metrics.csv"
df_metrics.to_csv(metrics_csv, index=False)
print(f"✅ Métricas salvas em: {metrics_csv}")

# 2. Copiar CSV para a pasta final do modelo
final_dir = "/content/drive/MyDrive/llama_unsloth_checkpoints/final_lora_block10"
os.system(f"cp {metrics_csv} {final_dir}/training_metrics.csv")
print(f"✅ CSV copiado para: {final_dir}/training_metrics.csv")

huggingface_token = userdata.get('huggingface_token')
login(token=huggingface_token)
repo_id = "guillherms/llama-3.2-3b-amazon-titles-100k-lora"
create_repo(repo_id, private=False, exist_ok=True)

repo_id = "guillherms/llama-3.2-3b-amazon-titles-100k-lora"

upload_folder(
    repo_id=repo_id,
    folder_path=final_dir,
    commit_message="Upload final do modelo + métricas (100k registros em 10 blocos)"
)

print(f"🚀 Modelo e métricas publicados em: https://huggingface.co/{repo_id}")

✅ Métricas salvas em: /content/drive/MyDrive/llama_unsloth_checkpoints/training_metrics.csv
✅ CSV copiado para: /content/drive/MyDrive/llama_unsloth_checkpoints/final_lora_block10/training_metrics.csv


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ra_block10/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...adapter_model.safetensors:   1%|1         |  599kB / 48.7MB            

🚀 Modelo e métricas publicados em: https://huggingface.co/guillherms/llama-3.2-3b-amazon-titles-100k-lora
